In [1]:
import pandas as pd
import numpy as np
from fastsdp_tools import get_project_path

In [2]:
SDPu_tab= pd.read_csv(get_project_path("results/benchmark/6x100-0.026/01_25_23h24_55s_run-SDPu-PP-RLT=100/results.csv"))
BB_tab = pd.read_csv(get_project_path("results/benchmark/mnist-6x100_0.026/2026-01-29_14-42-11_BB-alpha-beta-CROWN_test-bb/summary.csv"))
BB_tab["data_index"] = SDPu_tab["data_index"]

list_of_tab = {
    "SDPu" : SDPu_tab,
    "BB" : BB_tab,
}


In [3]:
for name, tab in list_of_tab.items():
    print(f"Processing table: {name}, number of columns : {len(tab.columns)}, number of rows : {len(tab)}")

Processing table: SDPu, number of columns : 31, number of rows : 100
Processing table: BB, number of columns : 10, number of rows : 100


In [4]:
SDPu_tab.columns

Index(['LAST_LAYER', 'MATRIX_BY_LAYERS', 'McC_betaz_logits', 'Nb_constraints',
       'Nb_stable_actives', 'Nb_stable_inactives', 'RLT', 'RLT_prop', 'Tij',
       'Tij_before_penultimate_layer', 'USE_STABLE_ACTIVES',
       'USE_STABLE_INACTIVES', 'beta_logits_comparaison_1',
       'beta_logits_comparaison_2', 'bound_time', 'data_index', 'dataset',
       'dual_obj_value', 'epsilon', 'iterations', 'label', 'label_predicted',
       'model', 'network', 'optimal_value', 'pretreatment_time',
       'primal_obj_value', 'status', 'target', 'time', 'triangularization'],
      dtype='object')

In [12]:
SDPu_tab[SDPu_tab['data_index']==99][['Nb_stable_actives', 'Nb_stable_inactives', 'bound_time', 'data_index', 'dataset','dual_obj_value', 'label', 'label_predicted','model', 'network', 'optimal_value', 'pretreatment_time','primal_obj_value', 'status', 'target', 'time']]

,Nb_stable_actives,Nb_stable_inactives,bound_time,data_index,dataset,dual_obj_value,label,label_predicted,model,network,optimal_value,pretreatment_time,primal_obj_value,status,target,time
99,0,287,1.681662,99,mnist,0.090153,9,0,UntargetedSDP,6x100,-0.104294,82.799973,-0.081041,0,NaN,3017.574988


In [ ]:
def traite_tableau_branch_and_bound(tab, name):

    print("Columns before renaming :", tab.columns)
    tab.rename(columns={col : name + "_" + col for col in tab.columns if col != "data_index"}, errors='raise', inplace=True)
    print("New columns :", tab.columns)

In [ ]:
def traite_tableau_sdp(tab, name):
    if "RLT_prop" in tab.columns:
        tab["RLT_prop"] = tab["RLT_prop"].fillna(0)
        RLT_prop = tab["RLT_prop"].unique()
        
        print("RLT_prop:", RLT_prop)
        #RLT_props = [0, 0.1, 0.3, 0.5]
        #tab = tab[tab["RLT_prop"].isin(RLT_props)].copy()

    tab["total_time"] = tab["time"] + tab["bound_time"]
    tab["robust"] = (tab["optimal_value"] >= 0) | (tab["status"]=="trivially_solved")
    tab["target"] = tab["target"].fillna("unknown")

    print("Columns before renaming :", tab.columns)
    tab.rename(columns={col : name+ "_" + col for col in tab.columns if col != "data_index"}, errors='raise', inplace=True)
    print("New columns :", tab.columns)

In [ ]:
for name, tab in list_of_tab.items():
    display(tab)
    if "Branch_and_Bound" in name:
        traite_tableau_branch_and_bound(tab, name)
    elif "SDP" in name:
        traite_tableau_sdp(tab, name)
    display(tab)

In [ ]:
merged = list_of_tab.popitem()[1]
for name, tab in list_of_tab.items():
    print(f"Table: {name}")
    print(tab.head())
    print(tab.columns)


    merged = pd.merge(merged, tab, left_on="data_index", right_on="data_index")

In [ ]:
merged.head()

In [ ]:
merged.columns

In [ ]:
list_of_tab["SDPu"]["data_index"]

In [ ]:
merged[merged["SDPu_robust"] & merged["total_falsified"]]

In [ ]:
merged[merged["SDPu_1_label"] != merged["SDPu_1_label_predicted"]][["data_index", "SDPu_01_optimal_value", "SDPu_1_optimal_value", "SDPu_01_total_time", "SDPu_1_total_time", "SDPu_1_label", "SDPu_1_label_predicted"]]

In [ ]:
merged[~merged["SDPu_robust"] & merged["BB_total_verified"]]

In [ ]:
merged[~merged["SDPu_FP_RLT=100_robust"] & merged["Branch_and_Bound_beta_CROWN_total_falsified"]]